# Loan Data Cleaning
Cleaning `loanP_train.csv` so it's ready for analysis or modeling.

In [ ]:
# pandas for the dataframe work, numpy for a couple of numeric helpers
import pandas as pd
import numpy as np

In [ ]:
# read the csv into a dataframe and peek at the first few rows
df = pd.read_csv('loanP_train.csv')
df.head()

In [ ]:
# rows x columns, plus each column's type and non-null count
# a quick way to spot which columns have gaps
print(df.shape)
df.info()

In [ ]:
# basic stats for the number columns, good for catching weird min/max values
df.describe()

In [ ]:
# check for repeated rows and repeated loan IDs
# each Loan_ID should only show up once
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate Loan_IDs:', df['Loan_ID'].duplicated().sum())

In [ ]:
# drop any duplicates anyway so the step is safe to rerun on new data
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
# count the missing values in each column
df.isnull().sum()

In [ ]:
# text columns: fill gaps with the most common value (mode)
# you can't take an average of 'Male' or 'Married', so mode is the go-to here
cat_cols = ['Gender', 'Marital_Status', 'Dependents', 'Self_Employed']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# LoanAmount is skewed by a few huge loans, so median is a safer fill than mean
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

# loan term and credit history are really categories (360 months, 1 or 0),
# so the most common value makes more sense than an average like 342.7
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

In [ ]:
# double check nothing is missing anymore
df.isnull().sum()

In [ ]:
# Dependents has '3+' as text, swap it for 3 so the column can be numeric
df['Dependents'] = df['Dependents'].replace('3+', '3').astype(int)
df['Dependents'].value_counts()

In [ ]:
# total household income is usually more useful than the two columns separately
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df[['ApplicantIncome', 'CoapplicantIncome', 'TotalIncome']].head()

In [ ]:
# turn yes/no style columns into 1/0 so models can read them
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Marital_Status'] = df['Marital_Status'].map({'Married': 1, 'Single': 0})
df['Graduate'] = df['Graduate'].map({'Yes': 1, 'No': 0})
df['Self_Employed'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# Property_Area has 3 options, so split it into separate 0/1 columns instead
df = pd.get_dummies(df, columns=['Property_Area'], dtype=int)
df.head()

In [ ]:
# final look at the cleaned data, then save it to a new file
df.info()
df.to_csv('loanP_train_clean.csv', index=False)